# `holspec` quickstart

This notebook demonstrates a small end-to-end workflow: generate noisy triangular lattice point clouds, run the `holspec` pipeline, analyze the resulting spectra, and create a few representative visualizations.

## 1. Setup

Import packages, configure paths, and create output directories for the quickstart workflow.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import yaml

%reload_ext autoreload
%autoreload 2

from holspec.analysis import EnsembleSpectraAnalysis
from holspec.paths import get_project_root, prepare_directory
from holspec.point_data import PointDataEnsemble, run_data_generation
from holspec.pipeline import run_pipeline, trace_provenance
from holspec.simplicial import SimplicialComplex
from holspec.utilities import get_keys_h5
from holspec.visualization import (
    format_axis,
    plot_distribution_lines,
    plot_eigval_distribution,
    plot_simplicial_complex_2d,
)

PROJECT_ROOT = get_project_root(Path.cwd())
EXAMPLES_DIR = PROJECT_ROOT / "examples"
CONFIGS_DIR = EXAMPLES_DIR / "configs"
DATA_DIR = EXAMPLES_DIR / "data"
RAW_DATA_DIR = DATA_DIR / "raw"
INTERIM_DATA_DIR = DATA_DIR / "interim"
OUTPUTS_DIR = EXAMPLES_DIR / "outputs" / "quickstart"

prepare_directory(CONFIGS_DIR, verbose=False)
DATA_GENERATION_CONFIG_PATH = CONFIGS_DIR / "data_generation_quickstart.yml"
PIPELINE_CONFIG_PATH = CONFIGS_DIR / "pipeline_quickstart.yml"

# Set to None to preserve generated artifacts between runs.
CLEAR_QUICKSTART_DIRS = "all"
for path in (RAW_DATA_DIR, INTERIM_DATA_DIR, OUTPUTS_DIR):
    prepare_directory(path, clear_mode=CLEAR_QUICKSTART_DIRS, verbose=False)

# Define figure settings for visualizations.
plt.rcParams["savefig.dpi"] = 200
plt.rcParams["savefig.bbox"] = "tight"

print(f"Project root: {PROJECT_ROOT}")
print("\nQuickstart directories:")
print(f"  Configs:      {CONFIGS_DIR.relative_to(PROJECT_ROOT)}")
print(f"  Data:         {DATA_DIR.relative_to(PROJECT_ROOT)}")
print(f"  Outputs:      {OUTPUTS_DIR.relative_to(PROJECT_ROOT)}")

Define small helper functions used throughout the quickstart.

In [ ]:
def format_noise_label(scale: float) -> str:
    return f"{scale:.2f}".replace(".", "p")


def make_noise_suffix(scale: float) -> str:
    if np.isclose(scale, 0.0):
        return "clean"
    return f"noise_{format_noise_label(scale)}"


def make_dataset_label(n_rings: int, noise_scale: float) -> str:
    return f"triangular_lattice_nr{n_rings}_{make_noise_suffix(noise_scale)}"


def make_pipeline_config(
    input_filepaths: dict[str, str],
    simplicial_constructions: dict,
    metric_models: dict,
    spectra_settings: dict,
) -> dict:
    quiet_runtime = {"verbose": False}

    return {
        "summary": {"created_by": "examples/quickstart.ipynb"},
        "inputs": {
            "data_dir": str(RAW_DATA_DIR.relative_to(PROJECT_ROOT)),
            "filepaths": input_filepaths,
        },
        "stages": {
            "topology_simplicial": {
                "configs": {"simplicial_constructions": simplicial_constructions},
                "runtime": quiet_runtime | {
                    "cache_incidence": False,
                    "validate_boundary_property": False,
                },
            },
            "geometry_metric": {
                "configs": {"metric_models": metric_models},
                "runtime": quiet_runtime | {"validate_metric": True},
            },
            "hodge_laplacian": {
                "configs": {},
                "runtime": quiet_runtime | {
                    "cache_laplacians": False,
                    "validate_laplacians": False,
                },
            },
            "spectra": {
                "configs": spectra_settings,
                "runtime": quiet_runtime,
            },
        },
        "runtime": {"verbose": False, "save_stage_configs": False},
        "outputs": {"data_dir": str(INTERIM_DATA_DIR.relative_to(PROJECT_ROOT))},
    }

## 2. Configure the example

Define the triangular lattice noise sweep and write the data generation config.

In [ ]:
# Define quickstart parameters.
CATEGORY = "quickstart"
N_RINGS = 10
NOISE_SCALES = [0.05, 0.25, 0.45]
NUM_REALIZATIONS = 1
BASE_SEED = 42

# Build the data generation config.
data_generation_config = {
    "summary": {
        "created_by": "examples/quickstart.ipynb",
    },
    "configs": {
        CATEGORY: {},
    },
    "runtime": {
        "verbose": False,
    },
    "outputs": {
        "stage_name": "point_data",
        "data_dir": str(RAW_DATA_DIR.relative_to(PROJECT_ROOT)),
        "category_subdirs": False,
    },
}

# Generate one dataset per noise level.
for noise_scale in NOISE_SCALES:
    label = make_dataset_label(N_RINGS, noise_scale)
    data_generation_config["configs"][CATEGORY][label] = {
        "base_config": {
            "generator": "trilatthex",
            "params": {
                "n_rings": N_RINGS,
            },
        },
        "noise_config": {
            "scale": noise_scale,
            "distribution": "normal",
        },
        "num_realizations": NUM_REALIZATIONS,
        "base_seed": BASE_SEED,
    }

# Store labels for later quickstart steps.
DATASET_LABELS = list(data_generation_config["configs"][CATEGORY])
DATASET_NOISE_SCALES = {
    make_dataset_label(N_RINGS, noise_scale): noise_scale
    for noise_scale in NOISE_SCALES
}

# Write the config to YAML.
with open(DATA_GENERATION_CONFIG_PATH, "w") as f:
    yaml.safe_dump(data_generation_config, f, sort_keys=False)

print(f"Data generation config written to {DATA_GENERATION_CONFIG_PATH.relative_to(PROJECT_ROOT)}")
print("\nDatasets to generate:")
for label in DATASET_LABELS:
    print(f"  {label}")

Define the pipeline settings and write the config used to run the quickstart workflow.

In [ ]:
# Point the pipeline to generated point cloud files.
pipeline_input_filepaths = {
    label: str((RAW_DATA_DIR / f"{label}.h5").relative_to(PROJECT_ROOT))
    for label in DATASET_LABELS
}

# Build simplicial complexes with Delaunay triangulation.
simplicial_constructions = {
    "delaunay": {
        "method": "delaunay",
        "params": {},
    },
}

# Use the combinatorial cochain metric.
metric_models = {
    "combinatorial": {
        "model": "combinatorial",
        "params": {},
    },
}

# Compute eigenvalue spectra with a dense solver.
spectra_settings = {
    "compute_spectra": True,
    "solver": "dense",
    "compute_eigenvectors": False,
}

# Build the pipeline config.
pipeline_config = make_pipeline_config(
    input_filepaths=pipeline_input_filepaths,
    simplicial_constructions=simplicial_constructions,
    metric_models=metric_models,
    spectra_settings=spectra_settings,
)

# Write the config to YAML.
with open(PIPELINE_CONFIG_PATH, "w") as f:
    yaml.safe_dump(pipeline_config, f, sort_keys=False)

print(f"Pipeline config written to {PIPELINE_CONFIG_PATH.relative_to(PROJECT_ROOT)}")
print("\nPipeline input files:")
for label, filepath in pipeline_input_filepaths.items():
    print(f"  {filepath}")

## 3. Generate point cloud data

Generate the point cloud ensembles from the quickstart data generation config.

In [ ]:
# Generate point cloud data.
generated_filepaths = run_data_generation(
    data_generation_config,
    project_root=PROJECT_ROOT,
    select_categories=[CATEGORY],
)
point_data_filepaths = generated_filepaths[CATEGORY]

print("\nGenerated point cloud data file paths:")
for label, filepath in point_data_filepaths.items():
    print(f"  {filepath.relative_to(PROJECT_ROOT)}")

## 4. Run the `holspec` pipeline

Construct simplicial complexes, define cochain metrics, assemble Hodge Laplacians, and compute eigenvalue spectra from the quickstart pipeline config.

In [ ]:
# Run the holspec pipeline.
pipeline_results = run_pipeline(
    pipeline_config,
    project_root=PROJECT_ROOT,
)
spectra_filepaths = pipeline_results["spectra"]

print("\nComputed spectra file paths:")
for label, output_files in spectra_filepaths.items():
    for output_label, filepath in output_files.items():
        print(f"  {filepath.relative_to(PROJECT_ROOT)}")

## 5. Analyze spectra

Configure the spectral analysis and analyze the eigenvalue spectra from the pipeline run.

In [ ]:
# Configure spectral analysis.
SPECTRA_OUTPUT_LABEL = "delaunay__combinatorial"
ANALYSIS_DEGREES = [0, 1, 2]
ANALYSIS_COMPONENTS = ("full",)
DISTRIBUTION_METHOD = "histogram"
DISTRIBUTION_NONZERO = True
DISTRIBUTION_PARAMS = {
    0: {"bins": 60, "range": (0, 12), "density": True},
    1: {"bins": 60, "range": (0, 12), "density": True},
    2: {"bins": 60, "range": (0, 6), "density": True},
}

# Analyze eigenvalue spectra.
analysis_keys = [
    (k, component)
    for k in ANALYSIS_DEGREES
    for component in ANALYSIS_COMPONENTS
]
spectra_analyses = {}
spectra_summaries = {}
spectra_distributions = {}

for label, output_files in spectra_filepaths.items():
    filepath = output_files[SPECTRA_OUTPUT_LABEL]
    
    # Read spectra from the pipeline output.
    esa = EnsembleSpectraAnalysis.from_file(
        filepath,
        project_root=PROJECT_ROOT,
        degrees=ANALYSIS_DEGREES,
        components=ANALYSIS_COMPONENTS,
    )
    file_analysis_keys = [
        (k, component)
        for k, component in analysis_keys
        if k in esa.degrees and component in esa.components
    ]

    spectra_analyses[label] = esa
    spectra_summaries[label] = {}
    spectra_distributions[label] = {}

    # Compute summaries and eigenvalue distributions.
    for k, component in file_analysis_keys:
        spectra_summaries[label][(k, component)] = esa.observables_summary(k, component)
        spectra_distributions[label][(k, component)] = esa.eigenvalue_distribution(
            k,
            component,
            nonzero=DISTRIBUTION_NONZERO,
            method=DISTRIBUTION_METHOD,
            method_params=DISTRIBUTION_PARAMS.get(k, {}),
        )

print("Analyzed spectra:")
for label, esa in spectra_analyses.items():
    noise_scale = DATASET_NOISE_SCALES[label]
    degrees = ", ".join(str(k) for k in esa.degrees)
    n_bins = next(iter(spectra_distributions[label].values()))["x"].size
    print(
        f"  noise={noise_scale:.2f}: "
        f"{esa.num_members} member(s), degrees {degrees}, "
        f"{n_bins}-bin nonzero distributions"
    )

Collect the per-noise distributions into a lightweight series for comparison plots.

In [ ]:
# Stack distributions across the noise sweep.
DISTRIBUTION_SERIES_LABELS = sorted(DATASET_LABELS, key=DATASET_NOISE_SCALES.get)
distribution_series_keys = [
    (k, component)
    for k, component in analysis_keys
    if all((k, component) in spectra_distributions[label] for label in DISTRIBUTION_SERIES_LABELS)
]

distribution_series = {
    "exp_param": "noise",
    "exp_values": np.array([DATASET_NOISE_SCALES[label] for label in DISTRIBUTION_SERIES_LABELS]),
    "analysis_keys": distribution_series_keys,
    "distribution_series": {},
}

for k, component in distribution_series_keys:
    x = spectra_distributions[DISTRIBUTION_SERIES_LABELS[0]][(k, component)]["x"]
    density_stack = np.array([
        spectra_distributions[label][(k, component)]["density_mean"]
        for label in DISTRIBUTION_SERIES_LABELS
    ])
    distribution_series["distribution_series"][(k, component)] = {
        "x": x,
        "density_stack": density_stack,
    }

print("Prepared distribution line comparison:")
print(f"  noise values: {distribution_series['exp_values']}")
print(f"  degrees: {', '.join(str(k) for k, _ in distribution_series_keys)}")

## 6. Visualize outputs

Plot selected simplicial complexes and eigenvalue distributions to visualize how the spectra respond to structural change.

In [ ]:
# Select datasets to visualize and compare.
VISUALIZATION_LABELS = [DATASET_LABELS[0], DATASET_LABELS[1], DATASET_LABELS[2]]

print("Visualizing datasets:")
for label in VISUALIZATION_LABELS:
    print(f"  {label} (noise={DATASET_NOISE_SCALES[label]:.2f})")

Plot the simplicial complexes constructed from the point cloud data.

In [ ]:
# Configure simplicial complex plots.
SIMPLICIAL_OUTPUT_LABEL = "delaunay"

simplex_colors = {0: "C0", 1: "C1", 2: "C2", 3: "C3"}
simplex_kwargs = {0: {"radius": 0.1}, 1: {"linewidth": 1.5}, 2: {}, 3: {}}
show_labels = False
show_orientation = False

axis_config_sc = {
    "aspect": "equal",
    "margins": (0.1, 0.1),
    "xlabel": "$x$",
    "ylabel": "$y$",
    "grid": True,
}

# Plot simplicial complexes for selected datasets.
fig, axes = plt.subplots(1, len(VISUALIZATION_LABELS), figsize=(5 * len(VISUALIZATION_LABELS), 5))
if len(VISUALIZATION_LABELS) == 1:
    axes = [axes]

for ax, label in zip(axes, VISUALIZATION_LABELS):
    filepath = pipeline_results["topology_simplicial"][label][SIMPLICIAL_OUTPUT_LABEL]
    member_key = sorted(k for k in get_keys_h5(filepath) if k.startswith("member_"))[0]
    member_index = int(member_key.split("_")[1])

    sc = SimplicialComplex.load(filepath, group=member_key, load_incidence=False)
    provenance = trace_provenance(filepath, PROJECT_ROOT)
    point_data_ensemble = PointDataEnsemble.load(provenance["point_data"])
    positions = point_data_ensemble[member_index].get_positions()

    plot_simplicial_complex_2d(
        sc.simplices,
        positions,
        simplex_colors=simplex_colors,
        simplex_kwargs=simplex_kwargs,
        show_labels=show_labels,
        show_orientation=show_orientation,
        ax=ax,
    )
    format_axis(ax, title=f"noise = {DATASET_NOISE_SCALES[label]:.2f}", **axis_config_sc)

fig.suptitle("Simplicial complexes", fontsize=11)
fig.tight_layout()

figpath = OUTPUTS_DIR / "simplicial_complexes.png"
fig.savefig(figpath)
print(f"Saved {figpath.relative_to(PROJECT_ROOT)}")

Plot the eigenvalue distributions and compare them across noise values.

In [ ]:
# Configure eigenvalue distribution plots.
eigval_distribution_config = {
    "degrees": ANALYSIS_DEGREES,
    "components": ANALYSIS_COMPONENTS,
    "nonzero": DISTRIBUTION_NONZERO,
    "distribution_params": DISTRIBUTION_PARAMS,
}

eigval_distribution_axis_config = {
    "xlabel": "Eigenvalue $\\lambda$",
    "ylabel": "Density",
    "grid": True,
    "grid_alpha": 0.2,
}

eigval_distribution_bar_config = {
    "alpha": 0.75,
    "edgecolor": "black",
}

# Match y-limits across datasets.
eigval_distribution_ylims = {}
for k, component in analysis_keys:
    max_density = max(
        np.nanmax(spectra_distributions[label][(k, component)]["density_mean"])
        for label in VISUALIZATION_LABELS
        if (k, component) in spectra_distributions[label]
    )
    if max_density > 0.0:
        eigval_distribution_ylims[(k, component)] = (0, max_density)

# Plot eigenvalue distributions for selected datasets.
for label in VISUALIZATION_LABELS:
    esa = spectra_analyses[label]
    file_degrees = [k for k in ANALYSIS_DEGREES if k in esa.degrees]
    file_config = {**eigval_distribution_config, "degrees": file_degrees}
    title = f"Eigenvalue distributions (noise = {DATASET_NOISE_SCALES[label]:.2f})"

    fig, axes = plot_eigval_distribution(
        esa,
        file_config,
        axis_config=eigval_distribution_axis_config,
        bar_config=eigval_distribution_bar_config,
        ylim_ranges=eigval_distribution_ylims,
        title=title,
        layout="horizontal",
        float_fmt=".4g",
    )

    suffix = make_noise_suffix(DATASET_NOISE_SCALES[label])
    figpath = OUTPUTS_DIR / f"eigval_distributions_{suffix}.png"
    fig.savefig(figpath)
    print(f"Saved {figpath.relative_to(PROJECT_ROOT)}")

Overlay the distribution lines to compare the full noise sweep in one visualization.

In [ ]:
# Configure distribution line comparison.
distribution_lines_axis_config = {
    "xlabel": "Eigenvalue $\\lambda$",
    "ylabel": "Density",
    "grid": True,
    "grid_alpha": 0.2,
    "title_fontsize": 9,
    "scilimits": (-2, 3),
}

DISTRIBUTION_LINES_CMAP = "viridis"
DISTRIBUTION_LINES_LINEWIDTH = 1.5

fig, axes = plot_distribution_lines(
    distribution_series,
    cmap=DISTRIBUTION_LINES_CMAP,
    linewidth=DISTRIBUTION_LINES_LINEWIDTH,
    mark_transition=False,
    axis_config=distribution_lines_axis_config,
    title="Eigenvalue distributions across noise levels",
    layout="horizontal",
)

figpath = OUTPUTS_DIR / "eigval_distribution_lines.png"
fig.savefig(figpath)
print(f"Saved {figpath.relative_to(PROJECT_ROOT)}")